# Coralscapes V2 on higher-quality 4K footage

Same pipeline as `coralscapes_v1.ipynb`, but using the **CoralscapesV2** SegFormer-B5 checkpoints (released with the CoralscapesV2 paper, ECCV 2026 Marine Vision workshop) instead of the V1 SegFormer-B2. In the project it was applied to the 4K DJI Osmo Action 5 Pro footage collected at Myora Reef, and assessed qualitatively.

| Variant | Hugging Face checkpoint | Classes |
| --- | --- | --- |
| `95` (default) | `EPFL-ECEO/segformer-b5-finetuned-coralscapesv2-1024-1024-95_class` | 95 fine-grained (+ index 0 = unlabelled) |
| `39` | `EPFL-ECEO/segformer-b5-finetuned-coralscapesv2-1024-1024` | legacy V1 label set |

Notes:
- Both checkpoints are public (no Hugging Face login needed). The DINOv3 + LoRA + DPT V2 models (`EPFL-ECEO/coralscapesv2-dinov3-vit{b,l}-lora-dpt[-95_class]`) are more accurate but gated and need a custom loader, so they are not used here.
- Class names are read from the model config or dataset files at runtime rather than hardcoded, because V2 has 95 classes rather than 39.
- Dataset: `josauder/coralscapesV2`. Project page: https://josauder.github.io/coralscapesv2
- No ground truth is used. V2 was only inspected qualitatively because the 4K footage was not annotated.

In [ ]:
import cv2
import numpy as np
import json
import torch
import torch.nn.functional as F
from PIL import Image
from huggingface_hub import hf_hub_download
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation

## Configuration

In [ ]:
# Config

VIDEO_PATH = "../data/bluerov_videos/Myora-DJI-4K.mp4" 
FRAME_SAMPLE_EVERY_N_SEC = 50

# Which CoralscapesV2 SegFormer-B5 checkpoint to use: 95 (fine-grained) or 39 (legacy V1 label set)
NUM_CLASSES_VARIANT = 95

MODEL_IDS = {
    95: "EPFL-ECEO/segformer-b5-finetuned-coralscapesv2-1024-1024-95_class",
    39: "EPFL-ECEO/segformer-b5-finetuned-coralscapesv2-1024-1024",
}
MODEL_ID = MODEL_IDS[NUM_CLASSES_VARIANT]
DATASET_ID = "josauder/coralscapesV2"

# Apple GPU (MPS) when available, otherwise CPU
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Model: {MODEL_ID}\nDevice: {device}")

## Load the preprocessor and model

In [ ]:
# Load preprocessor + model. The class count comes from the checkpoint itself.
preprocessor = SegformerImageProcessor.from_pretrained(MODEL_ID)
model = SegformerForSemanticSegmentation.from_pretrained(MODEL_ID).to(device)
model.eval()

NUM_CLASSES = model.config.num_labels   # includes index 0 (unlabelled)
print(f"num_labels in checkpoint: {NUM_CLASSES}")
print("Preprocessor size:", preprocessor.size)

## Class names and colours

In [ ]:
# ---------------------------------------------------------------------------
# Build id2label (and colours). Prefer the model config; fall back to the dataset's
# id2label.json; last resort is generic names. Colours come from the dataset's
# label2color.json when available, otherwise a deterministic random palette.
# ---------------------------------------------------------------------------
def load_json_from_hub(repo_id, filename, repo_type="dataset"):
    try:
        path = hf_hub_download(repo_id, filename, repo_type=repo_type)
        with open(path) as f:
            return json.load(f)
    except Exception as e:
        print(f"  (could not load {filename} from {repo_id}: {type(e).__name__})")
        return None

def normalise_id2label(d, n):
    """Return {int: str} with exactly n entries (adding 0='unlabelled' if it is the only gap), else None."""
    if not d:
        return None
    d = {int(k): str(v) for k, v in d.items()}
    if all(v.lower().startswith("label_") for v in d.values()):   # generic HF placeholder names
        return None
    if 0 not in d and len(d) == n - 1:
        d[0] = "unlabelled"
    return dict(sorted(d.items())) if len(d) == n else None

id2label = normalise_id2label(model.config.id2label, NUM_CLASSES)
if id2label is None:
    print("Model config has no usable id2label; trying dataset id2label.json")
    id2label = normalise_id2label(load_json_from_hub(DATASET_ID, "id2label.json"), NUM_CLASSES)
if id2label is None:
    print("WARNING: no label names found; using generic class_N names")
    id2label = {i: ("unlabelled" if i == 0 else f"class_{i}") for i in range(NUM_CLASSES)}

label2colour = load_json_from_hub(DATASET_ID, "label2color.json") or {}

rng = np.random.RandomState(0)
fallback_palette = rng.randint(30, 255, size=(NUM_CLASSES, 3))

id2colour = {}
for i, name in id2label.items():
    c = label2colour.get(name)
    id2colour[i] = tuple(int(x) for x in c[:3]) if c is not None else tuple(int(x) for x in fallback_palette[i])
id2colour[0] = (0, 0, 0)   # unlabelled/void

print(f"{len(id2label)} classes, {sum(n in label2colour for n in id2label.values())} colours from dataset")
for i, name in id2label.items():
    print(f"{i:3d}  {name}")

## Sliding-window inference

In [ ]:
# ---------------------------------------------------------------------------
# Sliding-window inference (official approach from the model card / repo)
# ---------------------------------------------------------------------------
def resize_image(image, target_size=1024):
    w_img, h_img = image.size
    if h_img < w_img:
        new_h, new_w = target_size, int(w_img * (target_size / h_img))
    else:
        new_h, new_w = int(h_img * (target_size / w_img)), target_size
    return image.resize((new_w, new_h))

def segment_image(image, crop_size=(1024, 1024), num_classes=None):
    num_classes = num_classes or NUM_CLASSES
    h_crop, w_crop = crop_size
    resized = resize_image(image, target_size=1024)
    img = torch.Tensor(np.array(resized).transpose(2, 0, 1)).unsqueeze(0)
    _, _, h_img, w_img = img.size()

    h_grids = int(np.round(3/2 * h_img / h_crop)) if h_img > h_crop else 1
    w_grids = int(np.round(3/2 * w_img / w_crop)) if w_img > w_crop else 1
    h_stride = int((h_img - h_crop + h_grids - 1) / (h_grids - 1)) if h_grids > 1 else h_crop
    w_stride = int((w_img - w_crop + w_grids - 1) / (w_grids - 1)) if w_grids > 1 else w_crop

    preds = torch.zeros((1, num_classes, h_img, w_img))
    count_mat = torch.zeros((1, 1, h_img, w_img))

    for h_idx in range(h_grids):
        for w_idx in range(w_grids):
            y1, x1 = h_idx * h_stride, w_idx * w_stride
            y2, x2 = min(y1 + h_crop, h_img), min(x1 + w_crop, w_img)
            y1, x1 = max(y2 - h_crop, 0), max(x2 - w_crop, 0)
            crop_img = img[:, :, y1:y2, x1:x2]

            with torch.no_grad():
                pil_crop = Image.fromarray(crop_img[0].numpy().transpose(1, 2, 0).astype(np.uint8))
                inputs = preprocessor(pil_crop, return_tensors="pt").to(device)
                outputs = model(**inputs)

            resized_logits = F.interpolate(
                outputs.logits, size=crop_img.shape[-2:], mode="bilinear", align_corners=False
            ).cpu()
            preds[:, :, y1:y2, x1:x2] += resized_logits
            count_mat[:, :, y1:y2, x1:x2] += 1

    preds = preds / count_mat
    preds = preds.argmax(dim=1)
    preds = F.interpolate(
        preds.unsqueeze(0).float(), size=(image.size[1], image.size[0]), mode="nearest"
    ).squeeze().numpy().astype(np.uint8)
    return preds

## Visualisation helpers

In [ ]:
import matplotlib.pyplot as plt

def colourise_mask(pred):
    """Convert (H, W) class-id array to an (H, W, 3) RGB colour image."""
    h, w = pred.shape
    colour_mask = np.zeros((h, w, 3), dtype=np.uint8)
    for class_id, colour in id2colour.items():
        colour_mask[pred == class_id] = colour
    return colour_mask

def show_segmentation(image, pred, alpha=0.6):
    colour_mask = colourise_mask(pred)
    image_np = np.array(image)
    overlay = (image_np * (1 - alpha) + colour_mask * alpha).astype(np.uint8)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(image_np); axes[0].set_title("Original"); axes[0].axis("off")
    axes[1].imshow(colour_mask); axes[1].set_title("Segmentation"); axes[1].axis("off")
    axes[2].imshow(overlay); axes[2].set_title("Overlay"); axes[2].axis("off")
    plt.tight_layout()
    plt.show()

## Extract frames from the video and segment each one

In [ ]:
# ---------------------------------------------------------------------------
# Extract frames from the video and segment each one
# ---------------------------------------------------------------------------
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise IOError(f"Could not open video: {VIDEO_PATH}")
fps = cap.get(cv2.CAP_PROP_FPS)
frame_interval = int(fps * FRAME_SAMPLE_EVERY_N_SEC)

frame_idx = 0
first_frame_checked = False

while True:
    ret, frame_bgr = cap.read()
    if not ret:
        break

    if frame_idx % frame_interval == 0:
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        image = Image.fromarray(frame_rgb)

        pred = segment_image(image)
        show_segmentation(image, pred)

        # One-time sanity check on class indexing (0-based vs 1-based)
        if not first_frame_checked:
            print(f"Predicted class id range: {pred.min()}-{pred.max()} (model has {NUM_CLASSES} outputs)")
            first_frame_checked = True

        labels, counts = np.unique(pred, return_counts=True)
        total_px = pred.size

        print(f"\n--- t={frame_idx/fps:.1f}s ---")
        for l, c in sorted(zip(labels, counts), key=lambda x: -x[1])[:5]:
            name = id2label.get(int(l), f"unlabelled/class_{l}")
            print(f"  {name:28s} {100*c/total_px:5.2f}%")

    frame_idx += 1

cap.release()